In [24]:
# Install dependencies (canonical: evaluation/requirements.txt)
%pip install -r ../../requirements.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
import kagglehub
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, precision_recall_fscore_support

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from pathlib import Path
from datasets import Dataset, DatasetDict
import time
from IPython.display import display

In [ ]:
# 1️⃣ Load dataset
path = kagglehub.dataset_download("amaanpoonawala/youtube-comments-sentiment-dataset")

#print("Path to dataset files:", path)

# Specify the directory path
directory_path = Path(path)

# List all files in the directory
files = [file.name for file in directory_path.iterdir() if file.is_file()]

# Print the list of files
#print("Files in directory:", files)

file_path = f'{path}/youtube_comments_cleaned.csv'

df = pd.read_csv(file_path)

# Clean nulls
df = df.dropna(subset=['VideoTitle', 'CommentText', 'Sentiment'])

# Normalize labels (make sure you have 0,1,2 mapping)
label_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df['Label'] = df['Sentiment'].map(label_map)

def build_text_with_context(row):
    return f"Video Title: {row['VideoTitle']}. Comment: {row['CommentText']}"

df['CommentTextWithContext'] = df.apply(build_text_with_context, axis=1)

Path to dataset files: /Users/guiavenas/.cache/kagglehub/datasets/amaanpoonawala/youtube-comments-sentiment-dataset/versions/1
Files in directory: ['youtube_comments_cleaned.csv']


In [27]:
results = []

In [ ]:
def execute_training_and_evaluation(dataset, x_column, model_name):
  # Split train / val / test using pandas before converting to Hugging Face Dataset
  texts = dataset[x_column].tolist()
  labels = dataset['Label'].tolist()

  train_texts, temp_texts, train_labels, temp_labels = train_test_split(
      texts, labels, test_size=0.2, random_state=42)

  val_texts, test_texts, val_labels, test_labels = train_test_split(
      temp_texts, temp_labels, test_size=0.5, random_state=42)

  # Convert splits to HF Datasets
  train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
  val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})
  test_dataset = Dataset.from_dict({'text': test_texts, 'label': test_labels})

  # Load tokenizer
  tokenizer = AutoTokenizer.from_pretrained(model_name)

  # Tokenization function
  def tokenize_function(examples):
      return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

  # Apply tokenization using map() - much more memory efficient
  train_dataset = train_dataset.map(tokenize_function, batched=True)
  val_dataset = val_dataset.map(tokenize_function, batched=True)
  test_dataset = test_dataset.map(tokenize_function, batched=True)

  # Set format for PyTorch
  train_dataset.set_format(type='numpy', columns=['input_ids', 'attention_mask', 'label'])
  train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
  val_dataset.set_format(type='numpy', columns=['input_ids', 'attention_mask', 'label'])
  val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
  test_dataset.set_format(type='numpy', columns=['input_ids', 'attention_mask', 'label'])
  test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

  # Load model
  model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

  # Define metrics
  def compute_metrics(pred):
      labels = pred.label_ids
      preds = np.argmax(pred.predictions, axis=1)
      acc = accuracy_score(labels, preds)
      f1 = f1_score(labels, preds, average='macro')

      precision, recall, f1_per_class, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0,1,2])

      return {
          "accuracy": acc,
          "macro_f1": f1,
          "precision_negative": precision[0],
          "precision_neutral": precision[1],
          "precision_positive": precision[2],
          "recall_negative": recall[0],
          "recall_neutral": recall[1],
          "recall_positive": recall[2],
          "f1_negative": f1_per_class[0],
          "f1_neutral": f1_per_class[1],
          "f1_positive": f1_per_class[2],
      }

  # Training arguments
  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      learning_rate=2e-5,
      per_device_train_batch_size=16,
      per_device_eval_batch_size=32,
      num_train_epochs=3,
      weight_decay=0.01,
      load_best_model_at_end=True,
      metric_for_best_model="macro_f1",  # Changed from "f1" to "macro_f1"
      save_total_limit=2,
      report_to="none" # Added to disable wandb logging
  )

  # Trainer
  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset,
      compute_metrics=compute_metrics,
  )

  # Train
  trainer.train()

  # Evaluate on validation set
  val_results = trainer.evaluate()
  #print("Validation Results:", val_results)

  # Evaluate on test set
  test_outputs = trainer.predict(test_dataset)
  test_labels_pred = np.argmax(test_outputs.predictions, axis=1)
  accuracy = accuracy_score(test_dataset['label'], test_labels_pred)
  f1 = f1_score(test_dataset['label'], test_labels_pred, average='macro')
  classification_report(test_dataset['label'], test_labels_pred)
  confusion_matrix(test_dataset['label'], test_labels_pred)
  
  test_metrics = compute_metrics(test_outputs)

  # Save model
  trainer.save_model("youtube_sentiment_model")
  tokenizer.save_pretrained("youtube_sentiment_model")

  return test_metrics, trainer.model, test_dataset

def run_rounds(n_rounds, sample_size, x_column, model_name):
    rounds_metrics = {
        "accuracy": [],
        "macro_f1": [],
        "precision_negative": [],
        "precision_neutral": [],
        "precision_positive": [],
        "recall_negative": [],
        "recall_neutral": [],
        "recall_positive": [],
        "f1_negative": [],
        "f1_neutral": [],
        "f1_positive": [],
        'per_sample_latency': []
    }
    
    for i in range(n_rounds):
        #print(f"Round {i+1} of {n_rounds}")
        used_df = df.sample(n=sample_size, random_state=i).reset_index(drop=True)
        test_metrics, model, test_dataset = execute_training_and_evaluation(used_df, x_column, model_name)
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()
        
        # Prepare data
        inputs = {
            'input_ids': test_dataset['input_ids'].to(device),
            'attention_mask': test_dataset['attention_mask'].to(device)
        }

        # Measure inference time
        start_time = time.time()
        with torch.no_grad():
            _ = model(**inputs)
        end_time = time.time()

        total_time = end_time - start_time
        per_sample_latency = total_time / sample_size
        
        for key in [r for r in rounds_metrics.keys() if r != 'per_sample_latency']:
            rounds_metrics[key].append(test_metrics[key])
        
        rounds_metrics['per_sample_latency'].append(per_sample_latency)
        
    r = {
        "n_rounds": n_rounds,
        "sample_size": sample_size,
        "x_column": x_column,
        "model_name": model_name,
    }
    
    for m in rounds_metrics.keys():
        r[f"mean_{m}"] = np.mean(rounds_metrics[m])
        r[f"std_{m}"] = np.std(rounds_metrics[m])
    
    #print("Round results:", r)
    
    results.append(r)

In [29]:
N_SAMPLES=2500
N_ROUNDS=10

In [30]:
#model_name = 'distilbert-base-uncased'
model_name = 'microsoft/deberta-v3-small'
#model_name = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
run_rounds(N_ROUNDS, N_SAMPLES, 'CommentText', model_name)

Round 1 of 10


/Users/guiavenas/source/repos/youtube-comment-reader/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/guiavenas/source/repos/youtube-comment-reader/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map: 100%|██████████| 250/250 [00:00<00:00, 19856.01 examples/s]
Some weights of DebertaV2ForSequenceClassification were no

{'eval_loss': 0.7463458180427551, 'eval_accuracy': 0.692, 'eval_macro_f1': 0.6901905096281563, 'eval_precision_negative': 0.7078651685393258, 'eval_precision_neutral': 0.64, 'eval_precision_positive': 0.7209302325581395, 'eval_recall_negative': 0.6847826086956522, 'eval_recall_neutral': 0.6153846153846154, 'eval_recall_positive': 0.775, 'eval_f1_negative': 0.6961325966850829, 'eval_f1_neutral': 0.6274509803921569, 'eval_f1_positive': 0.7469879518072289, 'eval_runtime': 0.7776, 'eval_samples_per_second': 321.486, 'eval_steps_per_second': 10.288, 'epoch': 1.0}


 67%|██████▋   | 250/375 [00:54<00:24,  5.05it/s]

{'eval_loss': 0.728818416595459, 'eval_accuracy': 0.708, 'eval_macro_f1': 0.7054617117117118, 'eval_precision_negative': 0.69, 'eval_precision_neutral': 0.6714285714285714, 'eval_precision_positive': 0.7625, 'eval_recall_negative': 0.75, 'eval_recall_neutral': 0.6025641025641025, 'eval_recall_positive': 0.7625, 'eval_f1_negative': 0.71875, 'eval_f1_neutral': 0.6351351351351351, 'eval_f1_positive': 0.7625, 'eval_runtime': 0.7503, 'eval_samples_per_second': 333.201, 'eval_steps_per_second': 10.662, 'epoch': 2.0}


...

{'train_runtime': 98.521, 'train_samples_per_second': 60.901, 'train_steps_per_second': 3.806, 'train_loss': 0.7336385904947916, 'epoch': 3.0}


100%|██████████| 8/8 [00:00<00:00, 12.35it/s]


Validation Results: {'eval_loss': 0.779861569404602, 'eval_accuracy': 0.664, 'eval_macro_f1': 0.661995413166744, 'eval_precision_negative': 0.64, 'eval_precision_neutral': 0.6323529411764706, 'eval_precision_positive': 0.7195121951219512, 'eval_recall_negative': 0.6956521739130435, 'eval_recall_neutral': 0.5308641975308642, 'eval_recall_positive': 0.7662337662337663, 'eval_f1_negative': 0.6666666666666666, 'eval_f1_neutral': 0.5771812080536913, 'eval_f1_positive': 0.7421383647798742, 'eval_runtime': 0.7818, 'eval_samples_per_second': 319.777, 'eval_steps_per_second': 10.233, 'epoch': 3.0}


100%|██████████| 8/8 [00:00<00:00, 12.35it/s]


Round results: {'n_rounds': 10, 'sample_size': 2500, 'x_column': 'CommentText', 'model_name': 'microsoft/deberta-v3-small', 'mean_accuracy': np.float64(0.6992), 'std_accuracy': np.float64(0.0200638979263751), 'mean_macro_f1': np.float64(0.6981892205253192), 'std_macro_f1': np.float64(0.017772525975721026), 'mean_precision_negative': np.float64(0.7005391713171021), 'std_precision_negative': np.float64(0.033640524274682095), 'mean_precision_neutral': np.float64(0.6417489263960363), 'std_precision_neutral': np.float64(0.04757480590288706), 'mean_precision_positive': np.float64(0.7675150446390856), 'std_precision_positive': np.float64(0.0373516179216908), 'mean_recall_negative': np.float64(0.7058592031765355), 'std_recall_negative': np.float64(0.07356172406975527), 'mean_recall_neutral': np.float64(0.6589343751134755), 'std_recall_neutral': np.float64(0.06390256048167707), 'mean_recall_positive': np.float64(0.7337032755176492), 'std_recall_positive': np.float64(0.05728402419332197), 'mean_

In [31]:
#model_name = 'distilbert-base-uncased'
model_name = 'microsoft/deberta-v3-small'
#model_name = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
run_rounds(N_ROUNDS, N_SAMPLES, 'CommentTextWithContext', model_name)

Round 1 of 10


/Users/guiavenas/source/repos/youtube-comment-reader/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/guiavenas/source/repos/youtube-comment-reader/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map: 100%|██████████| 250/250 [00:00<00:00, 14488.50 examples/s]
Some weights of DebertaV2ForSequenceClassification were no

{'eval_loss': 0.7694696187973022, 'eval_accuracy': 0.672, 'eval_macro_f1': 0.6664483162408986, 'eval_precision_negative': 0.6147540983606558, 'eval_precision_neutral': 0.6349206349206349, 'eval_precision_positive': 0.8153846153846154, 'eval_recall_negative': 0.8152173913043478, 'eval_recall_neutral': 0.5128205128205128, 'eval_recall_positive': 0.6625, 'eval_f1_negative': 0.7009345794392523, 'eval_f1_neutral': 0.5673758865248227, 'eval_f1_positive': 0.7310344827586207, 'eval_runtime': 0.7652, 'eval_samples_per_second': 326.722, 'eval_steps_per_second': 10.455, 'epoch': 1.0}


 67%|██████▋   | 250/375 [00:54<00:24,  5.12it/s]

{'eval_loss': 0.7275571227073669, 'eval_accuracy': 0.696, 'eval_macro_f1': 0.6894746489705558, 'eval_precision_negative': 0.6517857142857143, 'eval_precision_neutral': 0.7454545454545455, 'eval_precision_positive': 0.7228915662650602, 'eval_recall_negative': 0.7934782608695652, 'eval_recall_neutral': 0.5256410256410257, 'eval_recall_positive': 0.75, 'eval_f1_negative': 0.7156862745098039, 'eval_f1_neutral': 0.6165413533834586, 'eval_f1_positive': 0.7361963190184049, 'eval_runtime': 0.752, 'eval_samples_per_second': 332.448, 'eval_steps_per_second': 10.638, 'epoch': 2.0}


...

{'train_runtime': 121.3019, 'train_samples_per_second': 49.463, 'train_steps_per_second': 3.091, 'train_loss': 0.7689239908854166, 'epoch': 3.0}


100%|██████████| 8/8 [00:00<00:00, 12.56it/s]


Validation Results: {'eval_loss': 0.7725275754928589, 'eval_accuracy': 0.704, 'eval_macro_f1': 0.7035839278166957, 'eval_precision_negative': 0.7191011235955056, 'eval_precision_neutral': 0.6753246753246753, 'eval_precision_positive': 0.7142857142857143, 'eval_recall_negative': 0.6956521739130435, 'eval_recall_neutral': 0.6419753086419753, 'eval_recall_positive': 0.7792207792207793, 'eval_f1_negative': 0.7071823204419889, 'eval_f1_neutral': 0.6582278481012658, 'eval_f1_positive': 0.7453416149068323, 'eval_runtime': 0.7957, 'eval_samples_per_second': 314.196, 'eval_steps_per_second': 10.054, 'epoch': 3.0}


100%|██████████| 8/8 [00:00<00:00, 12.55it/s]


Round results: {'n_rounds': 10, 'sample_size': 2500, 'x_column': 'CommentTextWithContext', 'model_name': 'microsoft/deberta-v3-small', 'mean_accuracy': np.float64(0.6944), 'std_accuracy': np.float64(0.034278856457005666), 'mean_macro_f1': np.float64(0.6936666828410469), 'std_macro_f1': np.float64(0.03195453580145956), 'mean_precision_negative': np.float64(0.6928635331780244), 'std_precision_negative': np.float64(0.03799257908792443), 'mean_precision_neutral': np.float64(0.6287230796495686), 'std_precision_neutral': np.float64(0.0634574568930126), 'mean_precision_positive': np.float64(0.7751812768124664), 'std_precision_positive': np.float64(0.04219281319586405), 'mean_recall_negative': np.float64(0.7149899755008566), 'std_recall_negative': np.float64(0.07985872885795328), 'mean_recall_neutral': np.float64(0.6436046626676751), 'std_recall_neutral': np.float64(0.06101532934205503), 'mean_recall_positive': np.float64(0.7250124967546674), 'std_recall_positive': np.float64(0.055405387169821

In [32]:
model_name = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
run_rounds(N_ROUNDS, N_SAMPLES, 'CommentText', model_name)

Round 1 of 10


/Users/guiavenas/source/repos/youtube-comment-reader/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 250/250 [00:00<00:00, 22990.55 examples/s]
/Users/guiavenas/source/repos/youtube-comment-reader/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
 33%|███▎      | 125/375 [18:24<01:19,  3.16it/s]

{'eval_loss': 0.7040562629699707, 'eval_accuracy': 0.7, 'eval_macro_f1': 0.6947619047619048, 'eval_precision_negative': 0.6666666666666666, 'eval_precision_neutral': 0.6935483870967742, 'eval_precision_positive': 0.75, 'eval_recall_negative': 0.782608695652174, 'eval_recall_neutral': 0.5512820512820513, 'eval_recall_positive': 0.75, 'eval_f1_negative': 0.72, 'eval_f1_neutral': 0.6142857142857143, 'eval_f1_positive': 0.75, 'eval_runtime': 1.0511, 'eval_samples_per_second': 237.846, 'eval_steps_per_second': 7.611, 'epoch': 1.0}


 67%|██████▋   | 250/375 [19:10<00:39,  3.16it/s]

{'eval_loss': 0.7644454836845398, 'eval_accuracy': 0.736, 'eval_macro_f1': 0.7337139854318878, 'eval_precision_negative': 0.7087378640776699, 'eval_precision_neutral': 0.7246376811594203, 'eval_precision_positive': 0.782051282051282, 'eval_recall_negative': 0.7934782608695652, 'eval_recall_neutral': 0.6410256410256411, 'eval_recall_positive': 0.7625, 'eval_f1_negative': 0.7487179487179487, 'eval_f1_neutral': 0.6802721088435374, 'eval_f1_positive': 0.7721518987341772, 'eval_runtime': 1.0217, 'eval_samples_per_second': 244.701, 'eval_steps_per_second': 7.83, 'epoch': 2.0}


...

{'train_runtime': 610.3005, 'train_samples_per_second': 9.831, 'train_steps_per_second': 0.614, 'train_loss': 0.5632448323567708, 'epoch': 3.0}


100%|██████████| 8/8 [00:03<00:00,  2.62it/s]


Validation Results: {'eval_loss': 0.6897620558738708, 'eval_accuracy': 0.72, 'eval_macro_f1': 0.7174179292351254, 'eval_precision_negative': 0.7252747252747253, 'eval_precision_neutral': 0.6857142857142857, 'eval_precision_positive': 0.7415730337078652, 'eval_recall_negative': 0.717391304347826, 'eval_recall_neutral': 0.5925925925925926, 'eval_recall_positive': 0.8571428571428571, 'eval_f1_negative': 0.7213114754098361, 'eval_f1_neutral': 0.6357615894039735, 'eval_f1_positive': 0.7951807228915663, 'eval_runtime': 3.2321, 'eval_samples_per_second': 77.349, 'eval_steps_per_second': 2.475, 'epoch': 3.0}


100%|██████████| 8/8 [00:03<00:00,  2.05it/s]


Round results: {'n_rounds': 10, 'sample_size': 2500, 'x_column': 'CommentText', 'model_name': 'cardiffnlp/twitter-xlm-roberta-base-sentiment', 'mean_accuracy': np.float64(0.7192000000000001), 'std_accuracy': np.float64(0.027469255541423026), 'mean_macro_f1': np.float64(0.7173566776233985), 'std_macro_f1': np.float64(0.02769579296736495), 'mean_precision_negative': np.float64(0.7150473952423594), 'std_precision_negative': np.float64(0.048261616259532555), 'mean_precision_neutral': np.float64(0.671343051489744), 'std_precision_neutral': np.float64(0.05537608699111969), 'mean_precision_positive': np.float64(0.7772892188656411), 'std_precision_positive': np.float64(0.057727782288100424), 'mean_recall_negative': np.float64(0.7482004464560199), 'std_recall_negative': np.float64(0.048107930985448535), 'mean_recall_neutral': np.float64(0.6411140844109194), 'std_recall_neutral': np.float64(0.07201047750556423), 'mean_recall_positive': np.float64(0.7666977092175289), 'std_recall_positive': np.fl

In [33]:
model_name = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
run_rounds(N_ROUNDS, N_SAMPLES, 'CommentTextWithContext', model_name)

Round 1 of 10


/Users/guiavenas/source/repos/youtube-comment-reader/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 250/250 [00:00<00:00, 15458.65 examples/s]
/Users/guiavenas/source/repos/youtube-comment-reader/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
 33%|███▎      | 125/375 [03:24<06:52,  1.65s/it]

{'eval_loss': 0.6861469745635986, 'eval_accuracy': 0.712, 'eval_macro_f1': 0.7118198874296434, 'eval_precision_negative': 0.75, 'eval_precision_neutral': 0.627906976744186, 'eval_precision_positive': 0.7631578947368421, 'eval_recall_negative': 0.717391304347826, 'eval_recall_neutral': 0.6923076923076923, 'eval_recall_positive': 0.725, 'eval_f1_negative': 0.7333333333333333, 'eval_f1_neutral': 0.6585365853658537, 'eval_f1_positive': 0.7435897435897436, 'eval_runtime': 5.4333, 'eval_samples_per_second': 46.013, 'eval_steps_per_second': 1.472, 'epoch': 1.0}


 67%|██████▋   | 250/375 [06:47<03:35,  1.72s/it]

{'eval_loss': 0.7417318224906921, 'eval_accuracy': 0.716, 'eval_macro_f1': 0.7068950968251034, 'eval_precision_negative': 0.6578947368421053, 'eval_precision_neutral': 0.7647058823529411, 'eval_precision_positive': 0.7647058823529411, 'eval_recall_negative': 0.8152173913043478, 'eval_recall_neutral': 0.5, 'eval_recall_positive': 0.8125, 'eval_f1_negative': 0.7281553398058253, 'eval_f1_neutral': 0.6046511627906976, 'eval_f1_positive': 0.7878787878787878, 'eval_runtime': 5.2446, 'eval_samples_per_second': 47.668, 'eval_steps_per_second': 1.525, 'epoch': 2.0}


...

{'train_runtime': 317.3649, 'train_samples_per_second': 18.906, 'train_steps_per_second': 1.182, 'train_loss': 0.5880128580729167, 'epoch': 3.0}


100%|██████████| 8/8 [00:00<00:00,  9.21it/s]


Validation Results: {'eval_loss': 0.8326718211174011, 'eval_accuracy': 0.688, 'eval_macro_f1': 0.688083527071261, 'eval_precision_negative': 0.7011494252873564, 'eval_precision_neutral': 0.620253164556962, 'eval_precision_positive': 0.7380952380952381, 'eval_recall_negative': 0.6630434782608695, 'eval_recall_neutral': 0.6049382716049383, 'eval_recall_positive': 0.8051948051948052, 'eval_f1_negative': 0.6815642458100558, 'eval_f1_neutral': 0.6125, 'eval_f1_positive': 0.7701863354037267, 'eval_runtime': 1.023, 'eval_samples_per_second': 244.372, 'eval_steps_per_second': 7.82, 'epoch': 3.0}


100%|██████████| 8/8 [00:00<00:00,  9.31it/s]


Round results: {'n_rounds': 10, 'sample_size': 2500, 'x_column': 'CommentTextWithContext', 'model_name': 'cardiffnlp/twitter-xlm-roberta-base-sentiment', 'mean_accuracy': np.float64(0.7028), 'std_accuracy': np.float64(0.03220186330012596), 'mean_macro_f1': np.float64(0.7013670779892598), 'std_macro_f1': np.float64(0.03154606208662694), 'mean_precision_negative': np.float64(0.6976598240968328), 'std_precision_negative': np.float64(0.057972234318545715), 'mean_precision_neutral': np.float64(0.6426790449015218), 'std_precision_neutral': np.float64(0.04623810828468593), 'mean_precision_positive': np.float64(0.7759357322893908), 'std_precision_positive': np.float64(0.03133353614181474), 'mean_recall_negative': np.float64(0.7218940698001257), 'std_recall_negative': np.float64(0.07292268257703427), 'mean_recall_neutral': np.float64(0.6292156022994735), 'std_recall_neutral': np.float64(0.07486516184945602), 'mean_recall_positive': np.float64(0.7576328059265756), 'std_recall_positive': np.float

In [34]:
results_df = pd.DataFrame(results)
display(results_df)

,n_rounds,sample_size,x_column,model_name,mean_accuracy,std_accuracy,mean_macro_f1,std_macro_f1,mean_precision_negative,std_precision_negative,...,mean_recall_positive,std_recall_positive,mean_f1_negative,std_f1_negative,mean_f1_neutral,std_f1_neutral,mean_f1_positive,std_f1_positive,mean_per_sample_latency,std_per_sample_latency
0,10,2500,CommentText,microsoft/deberta-v3-small,0.6992,0.020064,0.698189,0.017773,0.700539,0.033641,...,0.733703,0.057284,0.700457,0.039144,0.645972,0.021237,0.748138,0.029905,0.003163,0.000894
1,10,2500,CommentTextWithContext,microsoft/deberta-v3-small,0.6944,0.034279,0.693667,0.031955,0.692864,0.037993,...,0.725012,0.055405,0.702102,0.051366,0.631342,0.030169,0.747557,0.035404,0.002885,0.000777
2,10,2500,CommentText,cardiffnlp/twitter-xlm-roberta-base-sentiment,0.7192,0.027469,0.717357,0.027696,0.715047,0.048262,...,0.766698,0.052517,0.729960,0.038035,0.652502,0.041560,0.769608,0.033765,0.004450,0.001319
3,10,2500,CommentTextWithContext,cardiffnlp/twitter-xlm-roberta-base-sentiment,0.7028,0.032202,0.701367,0.031546,0.697660,0.057972,...,0.757633,0.057447,0.706214,0.045260,0.632299,0.041008,0.765588,0.037156,0.005443,0.001052
